# 07 — Evaluate Metrics on DIODE Predictions

For each prediction:
1. Load raw MiDaS `.npy` prediction
2. Load DIODE depth and validity mask
3. Apply scale-and-shift alignment
4. Compute AbsRel, RMSE, δ1, etc.
5. Save results to `outputs/metrics/diode_results.csv`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'configs' / 'dataset_paths.yaml').exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / 'configs' / 'dataset_paths.yaml').exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
MODEL_TYPE = 'dpt_hybrid_384'
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'manifests' / 'diode_val_manifest.csv'
PRED_ROOT = PROJECT_ROOT / 'outputs' / 'predictions' / 'diode'
OUT_CSV = PROJECT_ROOT / 'outputs' / 'metrics' / 'diode_results.csv'

MIN_DEPTH = 1e-3
MAX_DEPTH = 300.0

In [ ]:
import cv2
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from src.datasets.transforms import load_diode_depth
from src.evaluation.align import align_scale_shift
from src.evaluation.metrics import compute_all_metrics

df = pd.read_csv(MANIFEST_PATH)
print(f'Evaluating {len(df)} DIODE samples')

In [ ]:
records = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred_path = PRED_ROOT / row['domain'] / row['scene'] / row['scan'] / f"{row['frame_id']}.npy"
    if not pred_path.exists():
        continue

    pred = np.load(str(pred_path))
    gt, valid_mask = load_diode_depth(row['depth_path'], row['mask_path'])
    if gt.shape != pred.shape:
        gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]), interpolation=cv2.INTER_NEAREST)
        valid_mask = cv2.resize(valid_mask.astype(np.uint8), (pred.shape[1], pred.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)

    aligned = align_scale_shift(pred, gt, valid_mask)
    metrics = compute_all_metrics(aligned, gt, valid_mask, MIN_DEPTH, MAX_DEPTH)

    records.append({
        'image_path': row['image_path'],
        'depth_path': row['depth_path'],
        'mask_path': row['mask_path'],
        'pred_path': str(pred_path),
        'dataset': 'diode',
        'corruption_type': row['corruption_type'],
        'severity': row['severity'],
        'domain': row['domain'],
        'scene': row['scene'],
        'scan': row['scan'],
        'frame_id': row['frame_id'],
        'model_name': MODEL_TYPE,
        **metrics,
    })

results_df = pd.DataFrame(records)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUT_CSV, index=False)
print(f'Saved {len(results_df)} results to {OUT_CSV}')

In [ ]:
results_df.groupby('domain')[['abs_rel', 'rmse', 'delta1']].mean().sort_values('abs_rel')